In [1]:
# Clear out the old directory completely
!rm -rf ./chroma_db
print("Old database cache cleared!")

Old database cache cleared!


In [2]:
%%capture
!pip install -U langchain transformers llama-index llama-index-readers-file llama-index-embeddings-openai sentence-transformers chromadb openai llama-index-vector-stores-chroma llama-index-embeddings-mistralai llama-index-llms-mistralai langchain-mistralai langchain-community

In [4]:
import os
from google.colab import userdata

# Safely fetch the key from Colab Secrets
os.environ["MISTRAL_API_KEY"] = userdata.get("MISTRAL_API_KEY")
print("Environment variables loaded successfully.")

Environment variables loaded successfully.


With the libraries installed, the next step is to generate or prepare the synthetic data: patient records and hospital policy documents. Would you like me to generate some placeholder code for this, or do you have existing files you'd like to load?

In [5]:
import pandas as pd
import numpy as np

# Create a synthetic healthcare dataset
np.random.seed(42)

num_patients = 1000

data = {
    'PatientID': range(1, num_patients + 1),
    'Age': np.random.randint(18, 90, num_patients),
    'Gender': np.random.choice(['Male', 'Female'], num_patients),
    'Diagnosis': np.random.choice([
        'Influenza', 'Hypertension', 'Diabetes', 'Asthma', 'Pneumonia',
        'Migraine', 'Arthritis', 'Bronchitis', 'Depression', 'Anxiety'
    ], num_patients),
    'AdmissionDate': pd.to_datetime('2023-01-01') + pd.to_timedelta(np.random.randint(0, 365, num_patients), unit='D'),
    'DischargeDate': pd.to_datetime('2023-01-01') + pd.to_timedelta(np.random.randint(5, 370, num_patients), unit='D'),
    'TreatmentCost': np.random.uniform(500, 15000, num_patients).round(2),
    'Doctor': np.random.choice(['Dr. Smith', 'Dr. Jones', 'Dr. Williams', 'Dr. Brown'], num_patients),
    'HospitalStayDays': np.random.randint(1, 30, num_patients)
}

healthcare_df = pd.DataFrame(data)

# Ensure DischargeDate is always after AdmissionDate
healthcare_df['DischargeDate'] = healthcare_df.apply(
    lambda row: row['AdmissionDate'] + pd.to_timedelta(row['HospitalStayDays'], unit='D')
    if row['DischargeDate'] <= row['AdmissionDate'] else row['DischargeDate'], axis=1
)

# Save to CSV
healthcare_df.to_csv('/content/healthcare_dataset.csv', index=False)
print("Synthetic 'healthcare_dataset.csv' generated.")


Synthetic 'healthcare_dataset.csv' generated.


In [6]:
import pandas as pd

# Load the synthetic patient records dataset
patient_df = pd.read_csv('/content/healthcare_dataset.csv')

print("Loaded patient records dataset:")
display(patient_df.head())

Loaded patient records dataset:


,PatientID,Age,Gender,Diagnosis,AdmissionDate,DischargeDate,TreatmentCost,Doctor,HospitalStayDays
0,1,69,Male,Bronchitis,2023-07-09,2023-12-20,7147.26,Dr. Jones,6
1,2,32,Male,Arthritis,2023-06-12,2023-07-05,11012.59,Dr. Brown,6
2,3,89,Female,Depression,2023-06-29,2023-07-21,8827.17,Dr. Smith,22
3,4,78,Male,Asthma,2023-02-03,2023-03-28,10174.58,Dr. Jones,29
4,5,38,Male,Hypertension,2023-02-10,2023-11-25,11767.21,Dr. Smith,12


Now that we have loaded the patient records, we also need to prepare the hospital policy documents for the RAG Agent. Since you don't have existing policy documents, I will generate some synthetic ones. We'll create a few markdown files with dummy policy content. Would that work for you?

### Generating Synthetic Hospital Policy Documents

For the RAG Agent, we'll create a few markdown files to simulate hospital policy documents. These will cover various topics like patient admission, discharge, and data privacy.

In [7]:
import os

policy_dir = 'hospital_policies'
os.makedirs(policy_dir, exist_ok=True)

policy_content_1 = """
# Patient Admission Policy

**Effective Date:** January 1, 2024

## Purpose
To outline the procedures for admitting patients to the hospital, ensuring a smooth and efficient process while maintaining patient safety and comfort.

## Scope
This policy applies to all hospital staff involved in the patient admission process, including administrative staff, nurses, and doctors.

## Procedure
1.  **Patient Registration:** Upon arrival, patients or their guardians must complete the registration forms, providing personal details, medical history, and insurance information.
2.  **Medical Assessment:** A preliminary medical assessment will be conducted by a registered nurse to evaluate the patient's immediate needs and assign a priority level.
3.  **Room Assignment:** Patients will be assigned to a suitable room based on their medical condition, gender, and availability.
4.  **Consent Forms:** Patients or their legal representatives must sign all necessary consent forms, including treatment consent and privacy policy acknowledgements.

## Responsibilities
*   **Admitting Staff:** Ensure all forms are correctly filled and processed.
*   **Nurses:** Conduct initial medical assessments and guide patients to their rooms.
*   **Doctors:** Review patient's medical history and treatment plans upon admission.

## Data Privacy
All patient information collected during admission is subject to strict data privacy regulations (e.g., HIPAA). Information will only be shared with authorized personnel directly involved in patient care.
"""

policy_content_2 = """
# Patient Discharge Policy

**Effective Date:** February 1, 2024

## Purpose
To ensure a safe and effective discharge process for all patients, facilitating continuity of care and preventing readmissions.

## Scope
This policy applies to all healthcare professionals involved in the patient discharge process.

## Procedure
1.  **Discharge Order:** A physician must issue a written discharge order.
2.  **Patient Education:** Patients and their families will receive comprehensive education regarding post-discharge care, medications, follow-up appointments, and potential warning signs.
3.  **Medication Reconciliation:** All medications will be reconciled, and prescriptions for new or adjusted medications will be provided.
4.  **Follow-up Arrangements:** Appointments for follow-up care with primary care physicians or specialists will be scheduled before discharge.
5.  **Transportation:** Assistance with transportation arrangements will be provided if needed.

## Responsibilities
*   **Physicians:** Issue discharge orders and approve discharge plans.
*   **Nurses:** Provide patient education and ensure medication reconciliation.
*   **Social Workers:** Assist with follow-up appointments and transportation.

## Patient Feedback
Patients will be encouraged to provide feedback on their discharge experience to help improve hospital services.
"""

policy_content_3 = """
# Data Privacy Policy (HIPAA Compliance)

**Effective Date:** March 1, 2024

## Purpose
To protect the privacy and security of patient health information (PHI) in compliance with the Health Insurance Portability and Accountability Act (HIPAA).

## Scope
This policy applies to all employees, volunteers, contractors, and business associates who have access to PHI.

## Core Principles
1.  **Confidentiality:** PHI must be kept confidential and accessed only by authorized personnel for treatment, payment, or healthcare operations (TPO).
2.  **Integrity:** PHI must be accurate and protected from unauthorized alteration or destruction.
3.  **Availability:** PHI must be accessible to authorized users when needed.

## Patient Rights
Patients have the right to:
*   Access their medical records.
*   Request amendments to their records.
*   Receive an accounting of disclosures of their PHI.
*   Request restrictions on certain uses and disclosures of their PHI.

## Employee Responsibilities
All employees must:
*   Complete annual HIPAA training.
*   Report any suspected breaches of PHI immediately.
*   Only access PHI that is necessary to perform their job duties.
*   Protect their login credentials and not share them.

## Penalties for Non-Compliance
Violations of this policy may result in disciplinary action, up to and including termination, and may also incur legal penalties as per HIPAA regulations.
"""

policies = {
    'admission_policy.md': policy_content_1,
    'discharge_policy.md': policy_content_2,
    'data_privacy_policy.md': policy_content_3,
}

for filename, content in policies.items():
    filepath = os.path.join(policy_dir, filename)
    with open(filepath, 'w') as f:
        f.write(content)
    print(f"Generated '{filepath}'")

print(f"\nSuccessfully generated {len(policies)} synthetic policy documents in the '{policy_dir}' directory.")
!ls -l {policy_dir}

Generated 'hospital_policies/admission_policy.md'
Generated 'hospital_policies/discharge_policy.md'
Generated 'hospital_policies/data_privacy_policy.md'

Successfully generated 3 synthetic policy documents in the 'hospital_policies' directory.
total 12
-rw-r--r-- 1 root root 1536 Jul 13 15:03 admission_policy.md
-rw-r--r-- 1 root root 1415 Jul 13 15:03 data_privacy_policy.md
-rw-r--r-- 1 root root 1354 Jul 13 15:03 discharge_policy.md


### 3. RAG Agent Implementation (Policy Documents)

Now we'll set up the Retrieval Augmented Generation (RAG) agent. This involves:
1.  Loading the policy documents.
2.  Splitting the documents into manageable chunks.
3.  Generating embeddings for these chunks.
4.  Storing the embeddings in a vector store.

In [8]:
from llama_index.core.readers.base import BaseReader
from llama_index.readers.file import MarkdownReader
from llama_index.core import Document
from llama_index.core.node_parser import SentenceSplitter
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.core import VectorStoreIndex, Settings
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.core.storage.storage_context import StorageContext
import chromadb

# 1. Document Loading
# Initialize the MarkdownReader to load our policy documents
loader = MarkdownReader()
policy_documents = loader.load_data(file='./hospital_policies/admission_policy.md')
policy_documents.extend(loader.load_data(file='./hospital_policies/discharge_policy.md'))
policy_documents.extend(loader.load_data(file='./hospital_policies/data_privacy_policy.md'))

print(f"Loaded {len(policy_documents)} policy documents.")
# Display a snippet of one document to verify
# print(policy_documents[0].text[:500] + '...')

Loaded 16 policy documents.


In [9]:
print("Confirming patient_df content:")
display(patient_df.head())

Confirming patient_df content:


,PatientID,Age,Gender,Diagnosis,AdmissionDate,DischargeDate,TreatmentCost,Doctor,HospitalStayDays
0,1,69,Male,Bronchitis,2023-07-09,2023-12-20,7147.26,Dr. Jones,6
1,2,32,Male,Arthritis,2023-06-12,2023-07-05,11012.59,Dr. Brown,6
2,3,89,Female,Depression,2023-06-29,2023-07-21,8827.17,Dr. Smith,22
3,4,78,Male,Asthma,2023-02-03,2023-03-28,10174.58,Dr. Jones,29
4,5,38,Male,Hypertension,2023-02-10,2023-11-25,11767.21,Dr. Smith,12


In [10]:
from llama_index.core.readers.base import BaseReader
from llama_index.readers.file import MarkdownReader
from llama_index.core import Document
from llama_index.core.node_parser import SentenceSplitter
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.core import VectorStoreIndex, Settings
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.core.storage.storage_context import StorageContext
import chromadb

# 1. Document Loading
# Initialize the MarkdownReader to load our policy documents
loader = MarkdownReader()
policy_documents = loader.load_data(file='/content/hospital_policies/admission_policy.md')
policy_documents.extend(loader.load_data(file='/content/hospital_policies/discharge_policy.md'))
policy_documents.extend(loader.load_data(file='/content/hospital_policies/data_privacy_policy.md'))

print(f"Loaded {len(policy_documents)} policy documents.")
# Display a snippet of one document to verify
# print(policy_documents[0].text[:500] + '...')

Loaded 16 policy documents.


In [11]:
# 2. Text Splitting (Node Parsing)
# Convert the loaded Document objects into 'Node' objects, which are smaller chunks.
splitter = SentenceSplitter(chunk_size=1024, chunk_overlap=20)
nodes = splitter.get_nodes_from_documents(policy_documents)

print(f"Created {len(nodes)} document nodes.")
# Display a snippet of one node to verify
# print(nodes[0].text[:500] + '...')

Created 16 document nodes.


In [12]:
import os
import chromadb
from llama_index.embeddings.mistralai import MistralAIEmbedding
from llama_index.core import VectorStoreIndex, Settings
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.core.storage.storage_context import StorageContext

# Initialize Embedding Model using the secure environment variable
embed_model = MistralAIEmbedding(
    model_name="mistral-embed",
    api_key=os.environ["MISTRAL_API_KEY"]
)

# Configure LlamaIndex globals
Settings.embed_model = embed_model
Settings.chunk_size = 1024

# Initialize ChromaDB
db = chromadb.PersistentClient(path="./chroma_db")
chroma_collection = db.get_or_create_collection(name="hospital_policies")
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

# Build Index from your nodes
index = VectorStoreIndex(
    nodes=nodes,
    storage_context=storage_context,
)

print("Embeddings generated and index created successfully with secure keys!")

Embeddings generated and index created successfully with secure keys!


In [13]:
import os

mistral_api_key_status = os.environ.get("MISTRAL_API_KEY")
if mistral_api_key_status:
    print(f"MISTRAL_API_KEY is set: {mistral_api_key_status[:5]}...{mistral_api_key_status[-5:]}")
else:
    print("MISTRAL_API_KEY is NOT set.")

MISTRAL_API_KEY is set: QPiYC...8EhJ5


In [14]:
import os
from getpass import getpass

if os.getenv("MISTRAL_API_KEY") is None:
    os.environ["MISTRAL_API_KEY"] = getpass("Enter your Mistral API key:")
    print("Mistral API key set (newly provided).")
else:
    print("Mistral API key already set.")
# Verify key is present without displaying the key
if os.getenv("MISTRAL_API_KEY"):
    print("Mistral API key is present in environment variables.")
else:
    print("Warning: Mistral API key is NOT present in environment variables.")

Mistral API key already set.
Mistral API key is present in environment variables.


### 4. RAG Agent Query Engine

In [15]:
# 4. RAG Agent Query Engine

from llama_index.llms.mistralai import MistralAI
from llama_index.core import Settings

# Initialize the Mistral LLM for RAG generation
mistral_llm = MistralAI(api_key=os.environ["MISTRAL_API_KEY"])
Settings.llm = mistral_llm

# Create a query engine from the index
query_engine = index.as_query_engine()

print("RAG Query Engine created successfully.")

RAG Query Engine created successfully.


Let's test the RAG Query Engine with a sample question about the admission policy.

In [16]:
from llama_index.llms.mistralai import MistralAI
from llama_index.core import Settings

# Re-initialize the Mistral LLM for generation (separate from embedding model)
# This is added here to ensure query_engine is defined if there was a kernel state issue.
mistral_llm = MistralAI(api_key=os.environ["MISTRAL_API_KEY"])
Settings.llm = mistral_llm

# Re-create a query engine from the index
# This is added here to ensure query_engine is defined if there was a kernel state issue.
query_engine = index.as_query_engine()

response = query_engine.query("What is the purpose of the patient admission policy?")
print("Query Response:")
print(response.response)

Query Response:
The purpose of the patient admission policy is to establish clear procedures for admitting patients to the hospital. This ensures the process is smooth, efficient, and prioritizes patient safety and comfort.


### 5. NLP-to-SQL Agent Implementation (Patient Records)

Now, let's set up the NLP-to-SQL agent using `langchain`. This agent will be responsible for converting natural language queries into SQL queries and executing them against our `patient_df` (which we'll represent as an in-memory SQLite database).

In [17]:
!pip install -q langchain
!pip install -q langchain-community
!pip install -q langchain-mistralai
!pip install -q sqlalchemy

In [18]:
import sqlalchemy
from sqlalchemy import create_engine

# Changed from :memory: to a local file database so the tables are stable
engine = create_engine("sqlite:///healthcare.db")
patient_df.to_sql("patient_records", engine, index=False, if_exists="replace")

print("Persistent SQLite database 'healthcare.db' successfully created on disk!")

Persistent SQLite database 'healthcare.db' successfully created on disk!


In [19]:
import os
from langchain_mistralai import ChatMistralAI
from langchain_community.agent_toolkits import SQLDatabaseToolkit
from langchain_community.agent_toolkits.sql.base import create_sql_agent
from langchain_community.utilities import SQLDatabase

# 1. Initialize LangChain Mistral Chat client with a retry buffer for 429 errors
mistral_llm_langchain = ChatMistralAI(
    api_key=os.environ["MISTRAL_API_KEY"],
    model="mistral-large-latest",
    max_retries=10
)

# 2. Connect to our new file-based database instance
db_sql = SQLDatabase(engine=engine)
toolkit = SQLDatabaseToolkit(db=db_sql, llm=mistral_llm_langchain)

# 3. Create the final worker SQL Agent
nlp_to_sql_agent = create_sql_agent(
    llm=mistral_llm_langchain,
    toolkit=toolkit,
    verbose=True
)

print("NLP-to-SQL Agent fully initialized against disk storage!")

NLP-to-SQL Agent fully initialized against disk storage!


Let's test the NLP-to-SQL Agent with a sample question about the patient records.

In [20]:
from sqlalchemy import create_engine

engine = create_engine("sqlite:///:memory:")

patient_df.to_sql(
    "patient_records",
    engine,
    index=False,
    if_exists="replace"
)

print("SQLite database created successfully.")

SQLite database created successfully.


In [21]:
import os

from langchain_mistralai import ChatMistralAI
from langchain_community.utilities import SQLDatabase
from langchain_community.agent_toolkits import SQLDatabaseToolkit
from langchain_community.agent_toolkits.sql.base import create_sql_agent

llm = ChatMistralAI(
    model="mistral-large-latest",
    api_key=os.environ["MISTRAL_API_KEY"]
)

db = SQLDatabase(engine)

toolkit = SQLDatabaseToolkit(
    db=db,
    llm=llm
)

nlp_to_sql_agent = create_sql_agent(
    llm=llm,
    toolkit=toolkit,
    verbose=True
)

print("SQL Agent created successfully.")

SQL Agent created successfully.


### 6. Orchestrator Agent Implementation

Now, we will implement the Orchestrator Agent. This agent's primary role is to act as a router, classifying incoming user queries and directing them to either the RAG Agent (for policy-related questions) or the NLP-to-SQL Agent (for patient record questions).

We will use LangChain's `RunnableBranch` and `ChatPromptTemplate` to achieve this routing logic.

In [23]:
import os
import time
import sqlalchemy
from sqlalchemy import create_engine
from langchain_mistralai import ChatMistralAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableBranch, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

print("Step 1: Locking database files securely to local disk...")
engine = create_engine("sqlite:///healthcare.db")
patient_df.to_sql("patient_records", engine, index=False, if_exists="replace")

# ADDED max_retries=10. This automatically absorbs 429 errors without crashing!
mistral_llm = ChatMistralAI(
    api_key=os.environ["MISTRAL_API_KEY"],
    model="mistral-large-latest",
    max_retries=10
)

print("Step 2: Building rate-limit protected routing pipeline...")
router_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant that classifies user queries."),
    ("human", "Classify the following query as either 'patient_data' or 'hospital_policy'.\nQuery: {query}\nClassification:")
])

def safe_route_classification(inputs):
    time.sleep(3) # Increased buffer
    return (router_prompt | mistral_llm | StrOutputParser()).invoke(inputs)

sql_generation_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a SQLite expert inside a healthcare system. Given an input question, write a syntactically correct SQLite query to run against the table 'patient_records'.
    The table contains columns: PatientID, Age, Gender, Diagnosis, AdmissionDate, DischargeDate, TreatmentCost, Doctor, HospitalStayDays.
    Output ONLY the raw SQL query string. Do not wrap it in markdown code blocks or add any conversational text."""),
    ("human", "{query}")
])

def execute_sql_and_summarize(query_str):
    time.sleep(3)
    generated_sql = (sql_generation_prompt | mistral_llm | StrOutputParser()).invoke({"query": query_str})
    clean_sql = generated_sql.strip().replace("```sql", "").replace("```", "")

    with engine.connect() as connection:
        sql_result = connection.execute(sqlalchemy.text(clean_sql))
        data_payload = sql_result.fetchall()

    time.sleep(3)
    summary_prompt = f"You are a helpful hospital assistant. Translate these raw database results: {data_payload} into a friendly natural language sentence answering the user question: '{query_str}'"
    final_answer = mistral_llm.invoke(summary_prompt)
    return final_answer.content

def invoke_llama_index_query_engine(query_dict):
    time.sleep(3)
    response = query_engine.query(query_dict['query'])
    return str(response)

# Connect lightweight branches
definitely_patient_data_branch = RunnableLambda(lambda x: execute_sql_and_summarize(x["query"]))
definitely_hospital_policy_branch = RunnableLambda(lambda x: invoke_llama_index_query_engine(x))

definitely_orchestrator = RunnableBranch(
    (lambda x: "patient_data" in x["topic"].lower(), definitely_patient_data_branch),
    (lambda x: "hospital_policy" in x["topic"].lower(), definitely_hospital_policy_branch),
    RunnableLambda(lambda x: mistral_llm.invoke(x["query"]).content)
)

orchestrator_agent_chain = (
    RunnablePassthrough.assign(topic=RunnableLambda(safe_route_classification))
    | definitely_orchestrator
)
print("System successfully assembled and protected!")

Step 1: Locking database files securely to local disk...
Step 2: Building rate-limit protected routing pipeline...
System successfully assembled and protected!


In [28]:
# Test 2: Ask a patient records question (SQL Generation + Execution)
patient_data_query = "What is the average age of patients diagnosed with Diabetes?"
print(f"\nUser Query (Patient Data): {patient_data_query}")
response_data = orchestrator_agent_chain.invoke({"query": patient_data_query})
print(f"Orchestrator Response (Patient Data): {response_data}")


User Query (Patient Data): What is the average age of patients diagnosed with Diabetes?
Orchestrator Response (Patient Data): Here’s a friendly and natural way to present the result:

*"The average age of patients diagnosed with Diabetes is approximately 54 years old."*
